In [10]:
from multiprocess import Pool
import itertools
import json
import numpy as np
from datasets import load_dataset

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from datasets import load_dataset

ds = load_dataset("mesolitica/pseudolabel-malaysian-youtube-whisper-large-v3-timestamp")

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
Generating combined split: 100%|██████████| 3085595/3085595 [00:02<00:00, 1149049.06 examples/s]


In [4]:
df = ds['combined'].to_pandas().to_dict(orient = 'records')

In [6]:
df = [(i, df[i]) for i in range(len(df))]
df[0]

(0,
 {'new_text': "<|startoftranscript|><|ms|><|transcribe|><|0.02|> Collab dia tak boleh kerja<|0.90|><|0.90|> Tak boleh<|1.40|><|1.40|> Kena ambil yang business class jugalah<|2.76|><|2.76|> Business class jugak<|3.50|><|3.50|> Faham faham faham<|4.66|><|4.66|> Macam tu<|5.48|><|5.48|> So gaji berbeza<|6.88|><|6.88|> Gaji berbeza<|8.52|><|8.52|> Antara<|8.78|><|8.78|> Kelas-kelas ni<|9.78|><|9.78|> Berbeza<|10.60|><|10.60|> Few thousand jugak lah<|12.60|><|12.60|> Jauh<|12.64|><|12.64|> Jauh beza dia<|13.40|><|13.40|> Beza seribu dua ribu<|14.94|><|14.94|> Macam tu lah<|15.54|><|15.54|> Total<|16.78|><|16.78|> Kalau macam Singapore<|18.78|><|18.78|> Dia macam tu juga<|19.50|><|19.50|> Tapi dia ikut qualification<|20.68|><|20.68|> You diploma business class<|22.04|><|22.04|> Ijazah<|22.98|><|22.98|> Macam tu eh<|23.64|><|23.64|> First class something like that<|24.76|><|24.76|> Dia tengok qualification<|26.22|><|26.22|> Let's say kan<|27.28|><|27.28|> Eh itu gaji gaji<|28.16|><|28.16|

In [9]:
df[1]

(1,
 {'new_text': "<|startoftranscript|><|en|><|transcribe|><|0.02|> If you don't have a job, you can't work.<|2.02|><|2.02|> You have to take business class.<|4.02|><|4.02|> I understand.<|6.02|><|6.02|> So, the salary is different?<|8.02|><|8.02|> The salary is different.<|10.02|><|10.02|> Between these classes?<|12.02|><|12.02|> It's different.<|14.02|><|14.02|> It's different.<|16.02|><|16.02|> It's like a thousand or two thousand.<|18.02|><|18.02|> If it's like Singapore, it's like that.<|20.02|><|20.02|> But they follow the qualifications.<|22.02|><|22.02|> You have a diploma, business class.<|24.02|><|24.02|> You get a degree.<|26.02|><|26.02|> Something like that.<|28.02|><|endoftext|>",
  'audio_filename': 'output-audio/2-0-0.mp3'})

In [14]:
from collections import defaultdict

data = defaultdict(list)
for d in df:
    data[d[1]['audio_filename']].append(d)

In [18]:
rows = list(data.values())

In [67]:
from tqdm import tqdm
import re

def loop(rows):
    rows, _ = rows

    selected = []
    for r in tqdm(rows):
        alignments = []
        scores = []
        s = []
        for r_ in r:
            try:
                with open(f'prepared-pseudolabel_alignment/{r_[0]}.alignment') as fopen:
                    d = json.load(fopen)
            except:
                continue
            alignments.append(d)
            scores.append(np.mean([d_['score'] for d_ in d]))
            s.append(r_)

        if len(scores) == 0:
            continue
            
        argmin = np.argmax(scores)

        r = s[argmin]
        d = alignments[argmin]

        text = re.sub(r"<\|.*?\|>", "", r[1]['new_text'])

        if len(text) > 600:
            continue

        if d[0]['start'] > 5:
            continue

        failed = False
        for i in range(len(d)):
            if i > 0 and (d[i]['start'] - d[i - 1]['end']) > 2:
                failed = True
                break

            if (d[i]['end'] - d[i]['start']) > 1.:
                failed = True
                break

        if failed:
            continue

        selected.append((r[1], d))
    return selected

In [68]:
filtered = loop((rows[:1], 0))

100%|██████████| 1/1 [00:00<00:00, 1739.65it/s]


In [72]:
filtered = multiprocessing(rows, loop, cores = 30)

100%|██████████| 65371/65371 [01:04<00:00, 1017.13it/s]


In [73]:
len(filtered) / len(rows), len(filtered), len(rows)

(0.17835102273915115, 349774, 1961155)

In [83]:
from collections import defaultdict
import os

audio_names = defaultdict(list)
for r in tqdm(filtered):
    i = os.path.split(r[0]['audio_filename'])[1].split('-')[:-1]
    i = '-'.join(i)
    audio_names[i].append(r)

100%|██████████| 349774/349774 [00:00<00:00, 591044.87it/s]


In [84]:
keys = list(audio_names.keys())
len(keys)

50055

In [89]:
keys = list(audio_names.keys())

group = []
for k in tqdm(keys):
    s = sorted(audio_names[k], key = lambda x: int(os.path.split(x[0]['audio_filename'])[-1].split('-')[1].replace('.mp3', '')))
    temp = [s[0]]
    previous = int(s[0][0]['audio_filename'].split('-')[-1].replace('.mp3', ''))
    final = False
    for s_ in s[1:]:
        i = int(s_[0]['audio_filename'].split('-')[-1].replace('.mp3', ''))
        if previous + 1 == i:
            final = False
            temp.append(s_)
            previous += 1
        else:
            final = True
            group.append(temp)
            temp = [s_]
            previous = i
    
    if not final:
        group.append(temp)

100%|██████████| 50055/50055 [00:00<00:00, 64741.24it/s]


In [90]:
len(group)

221271

In [92]:
group = [(i, group[i]) for i in range(len(group))]

In [93]:
# !rm -rf malaysian-whole malaysian-segment
!mkdir malaysian-whole
!mkdir malaysian-segment

In [94]:
import copy
import soundfile as sf
import librosa
import numpy as np

def loop(group):
    group, _ = group
    combine_all = []
    for g in tqdm(group):
        i = g[0]
        g = g[1]
        audio_files = []
        timestamps = []
        last_timestamp = 0
        for g_ in g:
            audio_files.append(g_[0]['audio_filename'])
            timestamp = copy.deepcopy(g_[1])
            for k in range(len(timestamp)):
                timestamp[k]['start'] += last_timestamp
                timestamp[k]['end'] += last_timestamp
            timestamps.extend(timestamp)
            last_timestamp = timestamp[-1]['end']
    
        word_level = []
        for t in timestamps:
            start = t['start']
            w = t['text']
            end = t['end']
            word_level.append(f"<|{start:.2f}|> {w}<|{end:.2f}|>")
        
        segments, temp = [], [timestamps[0]]
        last_t = timestamps[0]['end']
        for c_ in timestamps[1:]:
            if ((c_['start'] - last_t) > 0.4):
                segments.append(temp)
                temp = []
    
            last_t = c_['end']
            temp.append(c_)
    
        if len(temp):
            segments.append(temp)
    
        segment_level = []
        for s in segments:
            start = s[0]['start']
            end = s[-1]['end']
            w = ' '.join([c_['text'] for c_ in s])
            t = f"<|{start:.2f}|> {w}<|{end:.2f}|>"
            segment_level.append(t)
    
        y = [librosa.load(f, sr = 16000)[0] for f in audio_files]
        y = np.concatenate(y)
    
        audio_filename = f'malaysian-whole/{i}.mp3'
        sf.write(audio_filename, y, 16000)
    
        segment_audio_filenames = []
        streaming_word_level = []
        for k, s in enumerate(segments):
            segment_audio_filename = f'malaysian-segment/{i}-{k}.mp3'
            start = s[0]['start']
            end = s[-1]['end']
            y_ = y[int(start * 16000): int(end * 16000)]
            sf.write(segment_audio_filename, y_, 16000)
            segment_audio_filenames.append(segment_audio_filename)
    
            word_level_ = []
            for t in s:
                start = t['start']
                w = t['text']
                end = t['end']
                word_level_.append(f"<|{start:.2f}|> {w}<|{end:.2f}|>")
            streaming_word_level.append(''.join(word_level_))
            
        word_level = ''.join(word_level)
    
        combine_all.append({
            'mode': 'whole',
            'level': 'segment',
            'texts': [''.join(segment_level)],
            'audio_filenames': [audio_filename],
        })
        combine_all.append({
            'mode': 'whole',
            'level': 'word',
            'texts': [''.join(word_level)],
            'audio_filenames': [audio_filename],
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'segment',
            'texts': segment_level,
            'audio_filenames': segment_audio_filenames,
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'word',
            'texts': streaming_word_level,
            'audio_filenames': segment_audio_filenames,
        })
    return combine_all

In [95]:
combine_all = loop((group[:2], 0))

100%|██████████| 2/2 [00:02<00:00,  1.10s/it]


In [96]:
len(combine_all)

8

In [99]:
combine_all[0]

{'mode': 'whole',
 'level': 'segment',
 'texts': ["<|0.04|> Collab dia tak boleh kerja Tak boleh Kena ambil yang business class jugalah Business class jugak<|3.42|><|3.98|> Faham faham faham Macam tu So<|5.60|><|6.04|> gaji berbeza<|6.76|><|7.80|> Gaji berbeza Antara Kelas-kelas ni Berbeza<|10.40|><|11.70|> Few thousand jugak lah Jauh Jauh beza dia Beza seribu dua ribu Macam tu lah<|15.48|><|16.32|> Total<|16.72|><|17.86|> Kalau macam Singapore Dia macam tu juga Tapi dia ikut qualification You diploma business class<|21.98|><|22.44|> Ijazah Macam tu eh First class something like that<|24.64|><|25.44|> Dia tengok qualification<|26.34|><|26.78|> Let's say kan Eh itu gaji gaji Gaji gaji Oh itu gaji Serius eh Tapi kerja sama je If you want to Singapore airline I tak sure sangat lah<|34.04|><|35.20|> Tapi Singapore airline Tough juga sebenarnya<|37.28|><|37.96|> Tak silap dia punya training<|38.88|><|39.52|> Few months Few months jugalah<|40.76|><|42.12|> Okay Yang<|42.70|><|44.66|> You ker

In [102]:
# import IPython.display as ipd
# ipd.Audio(combine_all[0]['audio_filenames'][0])

In [103]:
combine_all = multiprocessing(group, loop, cores = 50)

IOPub message rate exceeded.23:31<20:43,  1.84it/s]  
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

100%|██████████| 4425/4425 [55:46<00:00,  1.32it/s]


In [104]:
len(combine_all)

885084

In [105]:
from datasets import Dataset

dataset = Dataset.from_list(combine_all)

In [106]:
dataset.push_to_hub('malaysia-ai/Malaysian-STT', 'malaysian')

Uploading the dataset shards: 100%|██████████| 4/4 [00:31<00:00,  7.75s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Malaysian-STT/commit/ff7fc42293dece2e4e583942cc934e766682e2e9', commit_message='Upload dataset', commit_description='', oid='ff7fc42293dece2e4e583942cc934e766682e2e9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Malaysian-STT', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Malaysian-STT'), pr_revision=None, pr_num=None)

In [107]:
!du -hs output-audio

328G	output-audio


In [108]:
!rm -rf output-audio